# LLMOps: Operating LLMs in Production

LLMOps adapts the MLOps discipline to the unique characteristics of Large Language Model systems. Traditional MLOps tracks model weights, compute, and accuracy. LLMOps must also track **prompts** (changing a prompt changes model behavior as much as retraining), **token costs** (every request has a per-token price), and **failure modes unique to LLMs** (hallucination, refusal, inconsistency).

This notebook covers the core LLMOps practices: prompt versioning, cost tracking, hallucination monitoring, LangFuse tracing, latency SLA monitoring, and model update management.

## What Makes LLMOps Different from MLOps

| Concern | Traditional MLOps | LLMOps |
|---|---|---|
| What changes | Model weights (retrain to change behavior) | Prompt text (edit to change behavior instantly) |
| Cost unit | Compute hours per training run | Tokens per request (ongoing) |
| Primary failure mode | Low accuracy on test set | Hallucination, refusal, off-topic response |
| Versioning | Model checkpoints | Prompt versions + upstream model version |
| Monitoring | Accuracy, data drift | Faithfulness, cost per token, latency p95 |

**Key insight**: In LLM systems, prompts are code. A prompt change is a deployment. Every prompt change must go through review, staging, testing, and rollout just like a code change. This is the central discipline of LLMOps.

In [1]:
import json
import datetime
import time
import random
import statistics
from typing import Optional

print("Core imports loaded")

Core imports loaded


## Section 1: Prompt Versioning

### Why Prompt Changes Are Deployments

Consider this: you have a customer support bot. You change the system prompt from *"Be helpful"* to *"Be helpful and always offer a refund if the customer seems frustrated"*. That single sentence change:

- Changes the bot's behavior on a subset of inputs (frustrated customers)
- May increase refund rates, affecting business metrics
- Cannot be detected by looking at code diffs (it lives in a database or config file)
- Will go unnoticed unless you have prompt version tracking

Prompt versioning means treating every prompt change with the same rigor as a code change: version numbers, changelogs, staged rollouts, and rollback capability.

In [2]:
# Prompt versioning with LangSmith
# LangSmith provides a Prompt Hub where teams can store, version, and pull prompts.
# In a real setup you would use:
#   from langsmith import Client
#   client = Client()
#   client.push_prompt("support-bot", object=prompt_template, tags=["v1.2"])
#   prompt = client.pull_prompt("support-bot:v1.2")

# We demonstrate the same pattern with a local registry

import yaml
import os
import pathlib

PROMPT_STORE_DIR = "/tmp/prompt_registry"
pathlib.Path(PROMPT_STORE_DIR).mkdir(parents=True, exist_ok=True)

class PromptRegistry:
    """Local prompt registry that versions prompts as YAML files.
    In production, back this with a database or LangSmith Hub."""

    def __init__(self, store_dir: str):
        self.store_dir = pathlib.Path(store_dir)
        self.store_dir.mkdir(parents=True, exist_ok=True)

    def _prompt_path(self, name: str, version: str) -> pathlib.Path:
        return self.store_dir / f"{name}_v{version}.yaml"

    def push_prompt(self, name: str, version: str, template: str,
                    changelog: str, author: str, tags: list = None):
        """Save a prompt version with metadata."""
        data = {
            "name": name,
            "version": version,
            "template": template,
            "changelog": changelog,
            "author": author,
            "tags": tags or [],
            "created_at": datetime.datetime.utcnow().isoformat(),
            "is_production": False,
        }
        path = self._prompt_path(name, version)
        with open(path, "w") as f:
            yaml.dump(data, f, default_flow_style=False)
        print(f"Pushed prompt '{name}' version {version}")
        return data

    def pull_prompt(self, name: str, version: str = "latest") -> dict:
        """Retrieve a prompt by name and version."""
        if version == "latest":
            versions = self.list_versions(name)
            if not versions:
                raise FileNotFoundError(f"No versions found for prompt '{name}'")
            version = versions[-1]

        path = self._prompt_path(name, version)
        if not path.exists():
            raise FileNotFoundError(f"Prompt '{name}' version {version} not found")
        with open(path) as f:
            return yaml.safe_load(f)

    def list_versions(self, name: str) -> list:
        """List all versions for a prompt, sorted."""
        files = list(self.store_dir.glob(f"{name}_v*.yaml"))
        versions = []
        for f in files:
            # Extract version from filename like 'support-bot_v1.0.yaml'
            ver = f.stem.split("_v", 1)[-1]
            versions.append(ver)
        return sorted(versions)

    def promote_to_production(self, name: str, version: str):
        """Mark a specific version as the production version."""
        # Demote current production version
        for ver in self.list_versions(name):
            path = self._prompt_path(name, ver)
            with open(path) as f:
                data = yaml.safe_load(f)
            if data.get("is_production"):
                data["is_production"] = False
                with open(path, "w") as f:
                    yaml.dump(data, f, default_flow_style=False)

        # Promote target version
        path = self._prompt_path(name, version)
        with open(path) as f:
            data = yaml.safe_load(f)
        data["is_production"] = True
        with open(path, "w") as f:
            yaml.dump(data, f, default_flow_style=False)
        print(f"Promoted '{name}' version {version} to production")

print("PromptRegistry class defined")

PromptRegistry class defined


In [3]:
# Demonstrate the prompt registry

registry = PromptRegistry(PROMPT_STORE_DIR)

# Push version 1.0
registry.push_prompt(
    name="support-bot",
    version="1.0",
    template="You are a helpful customer support assistant. Answer the user's question: {question}",
    changelog="Initial prompt",
    author="alice",
    tags=["support", "v1"],
)

# Push version 1.1 with a changelog explaining the change
registry.push_prompt(
    name="support-bot",
    version="1.1",
    template=(
        "You are a helpful customer support assistant for AcmeCorp. "
        "Be concise and friendly. If the user seems frustrated, acknowledge their frustration first. "
        "Answer the user's question: {question}"
    ),
    changelog="Added brand name, conciseness instruction, and frustration acknowledgment pattern. "
              "Expected to improve CSAT by ~5% based on A/B test results.",
    author="bob",
    tags=["support", "v1", "tested"],
)

# List versions
versions = registry.list_versions("support-bot")
print(f"\nAvailable versions: {versions}")

# Pull a specific version
prompt_data = registry.pull_prompt("support-bot", "1.1")
print(f"\nPulled prompt v1.1:")
print(f"  Template: {prompt_data['template'][:80]}...")
print(f"  Changelog: {prompt_data['changelog'][:80]}...")

# Promote to production
registry.promote_to_production("support-bot", "1.1")

prod_prompt = registry.pull_prompt("support-bot", "1.1")
print(f"  Is production: {prod_prompt['is_production']}")

Pushed prompt 'support-bot' version 1.0
Pushed prompt 'support-bot' version 1.1

Available versions: ['1.0', '1.1']

Pulled prompt v1.1:
  Template: You are a helpful customer support assistant for AcmeCorp. Be concise and friend...
  Changelog: Added brand name, conciseness instruction, and frustration acknowledgment patter...
Promoted 'support-bot' version 1.1 to production
  Is production: True


/tmp/ipykernel_18645/1628751073.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.datetime.utcnow().isoformat(),


## Section 2: Token Cost Tracking

LLM APIs charge per token. Unlike compute costs (which are batch and scheduled), LLM costs accumulate continuously with every user request. Cost overruns can be catastrophic: a bug causing 10x more tokens per request can 10x your monthly bill before morning.

**Best practice**: Track cost at every level -- per request, per user, per feature, per day. Set budget alerts.

In [4]:
# Token pricing constants (approximate, check official docs for current prices)
PRICING = {
    # Anthropic
    "claude-3-5-sonnet-20241022": {"input": 3.00 / 1_000_000, "output": 15.00 / 1_000_000},
    "claude-3-haiku-20240307":    {"input": 0.25 / 1_000_000, "output": 1.25  / 1_000_000},
    # OpenAI
    "gpt-4o":                     {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000},
    "gpt-4o-mini":                {"input": 0.15 / 1_000_000, "output": 0.60  / 1_000_000},
}

def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Calculate the cost of a single API call in USD."""
    if model not in PRICING:
        raise ValueError(f"Unknown model: {model}. Add to PRICING dict.")
    prices = PRICING[model]
    cost = input_tokens * prices["input"] + output_tokens * prices["output"]
    return cost

# Example calculation
cost = calculate_cost("claude-3-5-sonnet-20241022", input_tokens=500, output_tokens=200)
print(f"Cost for one Sonnet call (500 in, 200 out): ${cost:.6f}")

cost_haiku = calculate_cost("claude-3-haiku-20240307", input_tokens=500, output_tokens=200)
print(f"Cost for one Haiku call (500 in, 200 out):  ${cost_haiku:.6f}")
print(f"Cost ratio (Sonnet / Haiku): {cost / cost_haiku:.1f}x")

Cost for one Sonnet call (500 in, 200 out): $0.004500
Cost for one Haiku call (500 in, 200 out):  $0.000375
Cost ratio (Sonnet / Haiku): 12.0x


In [5]:
from collections import defaultdict

class CostTracker:
    """Track LLM costs per request, per user, per feature, per day.
    
    In production, persist this to a database (Postgres, ClickHouse, etc.)
    and expose it via a dashboard.
    """

    def __init__(self, daily_budget_usd: float = 50.0, alert_callback=None):
        self.daily_budget_usd = daily_budget_usd
        self.alert_callback = alert_callback or (lambda msg: print(f"[ALERT] {msg}"))
        # Each record: {timestamp, model, input_tokens, output_tokens, cost, user_id, feature}
        self.records: list = []

    def log_call(self, model: str, input_tokens: int, output_tokens: int,
                 user_id: str, feature: str, timestamp: datetime.datetime = None):
        """Log a single API call and its cost."""
        cost = calculate_cost(model, input_tokens, output_tokens)
        record = {
            "timestamp": (timestamp or datetime.datetime.utcnow()).isoformat(),
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost": cost,
            "user_id": user_id,
            "feature": feature,
        }
        self.records.append(record)

        # Check budget after logging
        today_cost = self.get_daily_cost()
        if today_cost > self.daily_budget_usd:
            self.alert_callback(
                f"Daily budget exceeded! Spent ${today_cost:.2f} of ${self.daily_budget_usd:.2f} budget."
            )
        elif today_cost > self.daily_budget_usd * 0.8:
            self.alert_callback(
                f"80% of daily budget used: ${today_cost:.2f} / ${self.daily_budget_usd:.2f}"
            )
        return cost

    def get_daily_cost(self, date: str = None) -> float:
        """Get total cost for a specific date (default: today)."""
        target = date or datetime.datetime.utcnow().strftime("%Y-%m-%d")
        return sum(
            r["cost"] for r in self.records
            if r["timestamp"].startswith(target)
        )

    def get_cost_by_user(self, date: str = None) -> dict:
        """Get cost breakdown by user_id for a date."""
        target = date or datetime.datetime.utcnow().strftime("%Y-%m-%d")
        costs = defaultdict(float)
        for r in self.records:
            if r["timestamp"].startswith(target):
                costs[r["user_id"]] += r["cost"]
        return dict(sorted(costs.items(), key=lambda x: x[1], reverse=True))

    def get_cost_by_feature(self, date: str = None) -> dict:
        """Get cost breakdown by feature for a date."""
        target = date or datetime.datetime.utcnow().strftime("%Y-%m-%d")
        costs = defaultdict(float)
        for r in self.records:
            if r["timestamp"].startswith(target):
                costs[r["feature"]] += r["cost"]
        return dict(sorted(costs.items(), key=lambda x: x[1], reverse=True))

print("CostTracker defined")

CostTracker defined


In [6]:
# Simulate a day's worth of LLM calls

tracker = CostTracker(daily_budget_usd=5.0)
today = datetime.datetime.utcnow().strftime("%Y-%m-%d")

random.seed(42)
users = ["user_001", "user_002", "user_003", "user_042"]
features = ["chat", "search", "summarization", "classification"]
models = ["claude-3-5-sonnet-20241022", "claude-3-haiku-20240307"]

# Simulate 80 calls throughout the day
for i in range(80):
    hour = random.randint(8, 22)
    ts = datetime.datetime.strptime(f"{today} {hour:02d}:00:00", "%Y-%m-%d %H:%M:%S")
    feature = random.choice(features)
    # Classification and search use cheaper/smaller model
    model = "claude-3-haiku-20240307" if feature in ["classification", "search"] else "claude-3-5-sonnet-20241022"
    tracker.log_call(
        model=model,
        input_tokens=random.randint(100, 1000),
        output_tokens=random.randint(50, 500),
        user_id=random.choice(users),
        feature=feature,
        timestamp=ts,
    )

print(f"\nDaily cost summary for {today}:")
print(f"  Total: ${tracker.get_daily_cost():.4f}")

print("\nCost by feature:")
for feature, cost in tracker.get_cost_by_feature().items():
    print(f"  {feature:<20} ${cost:.4f}")

print("\nCost by user (top 4):")
for user, cost in tracker.get_cost_by_user().items():
    print(f"  {user:<15} ${cost:.4f}")


Daily cost summary for 2026-06-29:
  Total: $0.2657

Cost by feature:
  chat                 $0.1581
  summarization        $0.0897
  search               $0.0094
  classification       $0.0085

Cost by user (top 4):
  user_003        $0.1068
  user_001        $0.0766
  user_002        $0.0509
  user_042        $0.0314


/tmp/ipykernel_18645/1354324521.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.datetime.utcnow().strftime("%Y-%m-%d")
/tmp/ipykernel_18645/327787434.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  target = date or datetime.datetime.utcnow().strftime("%Y-%m-%d")
/tmp/ipykernel_18645/327787434.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  target = date or datetime.datetime.utcnow().strftime("%Y-%m-%d")
/tmp/ipykernel_18645/327787434.py:53: DeprecationWarning: datetime.datetime.utcnow() 

## Section 3: Hallucination Monitoring

Hallucination is the LLM-specific failure mode: the model generates confident, fluent text that is factually wrong. Unlike traditional model errors (low accuracy on a test set), hallucinations are:

- Hard to detect automatically (the output looks grammatically correct)
- Potentially high-stakes (a hallucinated drug dosage or legal clause can cause real harm)
- System-specific (what counts as a hallucination depends on what the ground truth is)

**What to monitor**:
- **Faithfulness**: for RAG, is every claim in the answer supported by retrieved context?
- **Citation accuracy**: do cited sources actually say what the model claims?
- **Refusal rate**: how often does the model refuse to answer (may indicate over-filtering)
- **Toxicity rate**: offensive output rate

In [7]:
# Faithfulness check for RAG systems
# For each claim in the answer, check if it is supported by the retrieved context.
# Simple heuristic: check if key phrases from the answer appear in the context.

def simple_faithfulness_check(answer: str, context: str) -> dict:
    """Simple word-overlap faithfulness check.
    
    For production, replace with an LLM-as-judge (see next cell).
    This heuristic is fast but coarse: it checks if significant
    noun phrases from the answer appear in the context.
    """
    # Tokenize to words, ignoring stopwords
    stopwords = {"the", "a", "an", "is", "are", "was", "were", "be", "been",
                 "to", "of", "and", "in", "it", "for", "on", "with", "this", "that"}
    answer_words = set(w.lower().strip(".,!?") for w in answer.split() if w.lower() not in stopwords)
    context_words = set(w.lower().strip(".,!?") for w in context.split() if w.lower() not in stopwords)

    if not answer_words:
        return {"faithfulness_score": 1.0, "supported": True, "method": "empty_answer"}

    overlap = answer_words & context_words
    faithfulness_score = len(overlap) / len(answer_words)

    return {
        "faithfulness_score": round(faithfulness_score, 3),
        "supported": faithfulness_score >= 0.5,
        "answer_words": len(answer_words),
        "overlap_words": len(overlap),
        "method": "word_overlap",
    }

# Test with a RAG example
context = """
AcmeCorp was founded in 2010 by Jane Smith. The company is headquartered in San Francisco.
In 2023, AcmeCorp reported revenue of $45 million and employs 320 people.
"""

faithful_answer = "AcmeCorp was founded in 2010 by Jane Smith and has 320 employees."
hallucinated_answer = "AcmeCorp was founded in 2015 and had an IPO in 2022, raising $200 million."

print("Faithful answer check:")
print(simple_faithfulness_check(faithful_answer, context))

print("\nHallucinated answer check:")
print(simple_faithfulness_check(hallucinated_answer, context))

Faithful answer check:
{'faithfulness_score': 0.778, 'supported': True, 'answer_words': 9, 'overlap_words': 7, 'method': 'word_overlap'}

Hallucinated answer check:
{'faithfulness_score': 0.333, 'supported': False, 'answer_words': 9, 'overlap_words': 3, 'method': 'word_overlap'}


In [8]:
# LLM-as-Judge for faithfulness
# Use a smaller/cheaper model to evaluate the output of the main model.
# This scales: you do not need humans to check every output.

FAITHFULNESS_JUDGE_PROMPT = """
You are a faithfulness evaluator. Your job is to determine whether a given answer
is fully supported by the provided context.

Context:
{context}

Answer:
{answer}

Evaluate: Is every factual claim in the answer directly supported by the context?
Respond with a JSON object with these fields:
- faithful: true or false
- score: float between 0.0 (not faithful) and 1.0 (fully faithful)
- reasoning: one sentence explaining your verdict
- unsupported_claims: list of claims not found in context (empty list if faithful)
"""

def llm_judge_faithfulness(answer: str, context: str) -> dict:
    """Use an LLM to evaluate faithfulness. In production, call a real API.
    Here we return a mock response to illustrate the pattern."""

    # MOCK: in production, this calls claude-3-haiku or gpt-4o-mini
    # response = client.messages.create(
    #     model="claude-3-haiku-20240307",
    #     max_tokens=300,
    #     messages=[{"role": "user", "content": prompt}]
    # )
    # return json.loads(response.content[0].text)

    # Mock response for demonstration
    if "IPO" in answer or "2015" in answer:
        return {
            "faithful": False,
            "score": 0.1,
            "reasoning": "The answer claims an IPO in 2022 and founding in 2015, neither of which appear in the context.",
            "unsupported_claims": ["founded in 2015", "IPO in 2022", "raised $200 million"],
        }
    else:
        return {
            "faithful": True,
            "score": 0.95,
            "reasoning": "All claims (founding year, founder name, employee count) are present in the context.",
            "unsupported_claims": [],
        }

print("LLM-as-Judge for faithful answer:")
result = llm_judge_faithfulness(faithful_answer, context)
print(json.dumps(result, indent=2))

print("\nLLM-as-Judge for hallucinated answer:")
result = llm_judge_faithfulness(hallucinated_answer, context)
print(json.dumps(result, indent=2))

LLM-as-Judge for faithful answer:
{
  "faithful": true,
  "score": 0.95,
  "reasoning": "All claims (founding year, founder name, employee count) are present in the context.",
  "unsupported_claims": []
}

LLM-as-Judge for hallucinated answer:
{
  "faithful": false,
  "score": 0.1,
  "reasoning": "The answer claims an IPO in 2022 and founding in 2015, neither of which appear in the context.",
  "unsupported_claims": [
    "founded in 2015",
    "IPO in 2022",
    "raised $200 million"
  ]
}


In [9]:
# Hallucination monitor: track faithfulness scores over time

class HallucinationMonitor:
    """Track faithfulness scores and alert when hallucination rate is high."""

    def __init__(self, alert_threshold: float = 0.2):
        """alert_threshold: alert if more than this fraction of responses are unfaithful."""
        self.alert_threshold = alert_threshold
        self.records = []

    def log(self, query: str, answer: str, context: str, faithfulness_score: float):
        self.records.append({
            "timestamp": datetime.datetime.utcnow().isoformat(),
            "query": query[:100],
            "faithfulness_score": faithfulness_score,
            "is_faithful": faithfulness_score >= 0.7,
        })

    def hallucination_rate(self, last_n: int = 100) -> float:
        recent = self.records[-last_n:]
        if not recent:
            return 0.0
        unfaithful = sum(1 for r in recent if not r["is_faithful"])
        return unfaithful / len(recent)

    def check_alert(self, last_n: int = 100) -> Optional[str]:
        rate = self.hallucination_rate(last_n)
        if rate > self.alert_threshold:
            return (
                f"[HALLUCINATION ALERT] Rate={rate:.1%} exceeds threshold={self.alert_threshold:.1%} "
                f"over last {min(len(self.records), last_n)} requests."
            )
        return None

monitor = HallucinationMonitor(alert_threshold=0.2)

# Simulate mixed results
random.seed(0)
for i in range(50):
    score = random.uniform(0.3, 1.0) if random.random() > 0.25 else random.uniform(0.0, 0.5)
    monitor.log(f"query_{i}", f"answer_{i}", "context", score)

rate = monitor.hallucination_rate()
print(f"Hallucination rate over last 50 requests: {rate:.1%}")

alert = monitor.check_alert()
if alert:
    print(alert)
else:
    print("No hallucination alert: rate within acceptable range.")

Hallucination rate over last 50 requests: 52.0%
[HALLUCINATION ALERT] Rate=52.0% exceeds threshold=20.0% over last 50 requests.


/tmp/ipykernel_18645/6819494.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.datetime.utcnow().isoformat(),


## Section 4: LangFuse Integration

LangFuse is open-source LLM observability. It traces every LLM call: what went in, what came out, how long it took, how many tokens were used, and what it cost. The data flows into a dashboard where you can:

- See traces for individual requests (debug failures)
- Aggregate metrics over time (cost trends, latency p95)
- Score outputs (human feedback, automated evals)
- Tag and filter by user, session, feature

LangFuse works with any LLM (Anthropic, OpenAI, open-source). You can self-host it or use the cloud service.

In [10]:
# LangFuse setup and basic tracing
# To run for real:
#   1. pip install langfuse
#   2. Set LANGFUSE_SECRET_KEY and LANGFUSE_PUBLIC_KEY env vars
#   3. Set LANGFUSE_HOST (defaults to https://cloud.langfuse.com)

try:
    from langfuse import Langfuse
    from langfuse.decorators import observe, langfuse_context
    LANGFUSE_AVAILABLE = True
    print("LangFuse imported successfully")
except ImportError:
    LANGFUSE_AVAILABLE = False
    print("LangFuse not configured (no API keys). Showing code pattern only.")

# The @observe decorator automatically traces the function call
# Every call creates a trace in the LangFuse dashboard

if LANGFUSE_AVAILABLE:
    langfuse = Langfuse()  # reads keys from env vars

    @observe()
    def rag_pipeline(question: str, user_id: str) -> str:
        """RAG pipeline with automatic LangFuse tracing."""
        # LangFuse creates a parent trace for this function call
        # Each sub-call (retrieval, generation) becomes a child span

        # Tag the trace with user context
        langfuse_context.update_current_trace(
            user_id=user_id,
            tags=["rag", "production"],
        )

        retrieved_context = retrieve_docs(question)   # traced span
        answer = generate_answer(question, retrieved_context)  # traced span
        return answer

    @observe()
    def retrieve_docs(question: str) -> str:
        """Retrieval step (becomes a span in the trace)."""
        return "AcmeCorp was founded in 2010 by Jane Smith."

    @observe()
    def generate_answer(question: str, context: str) -> str:
        """Generation step (becomes a span in the trace)."""
        return f"Based on the context: AcmeCorp was founded in 2010."

else:
    # Show the pattern without running it
    print()
    print("LangFuse tracing pattern:")
    print()
    print("  @observe()")
    print("  def rag_pipeline(question: str, user_id: str) -> str:")
    print("      langfuse_context.update_current_trace(user_id=user_id)")
    print("      context = retrieve_docs(question)   # auto-traced")
    print("      answer  = generate_answer(question, context)  # auto-traced")
    print("      return answer")
    print()
    print("Each @observe() call creates a span. Traces are visible in the LangFuse dashboard.")

LangFuse not configured (no API keys). Showing code pattern only.

LangFuse tracing pattern:

  @observe()
  def rag_pipeline(question: str, user_id: str) -> str:
      langfuse_context.update_current_trace(user_id=user_id)
      context = retrieve_docs(question)   # auto-traced
      answer  = generate_answer(question, context)  # auto-traced
      return answer

Each @observe() call creates a span. Traces are visible in the LangFuse dashboard.


In [11]:
# Manual trace creation in LangFuse
# For cases where you need more control than the @observe decorator

manual_trace_example = """
# Manual trace creation in LangFuse

from langfuse import Langfuse
langfuse = Langfuse()

# Create a trace for a user session
trace = langfuse.trace(
    name="customer-support-chat",
    user_id="user_001",
    session_id="sess_abc123",
    tags=["support", "production"],
    input={"question": "How do I reset my password?"},
)

# Create a span for retrieval
retrieval_span = trace.span(
    name="vector-retrieval",
    input={"query": "password reset"},
)
docs = retrieve_documents("password reset")
retrieval_span.end(output={"num_docs": len(docs)})

# Create a generation span with token counts
generation = trace.generation(
    name="claude-generation",
    model="claude-3-5-sonnet-20241022",
    input=[{"role": "user", "content": question}],
    usage={"input": 350, "output": 150},  # tokens
)
answer = call_llm(question, docs)
generation.end(output=answer)

# Close the trace
trace.update(output={"answer": answer})
langfuse.flush()  # send to LangFuse server
"""

print("Manual LangFuse trace pattern:")
print(manual_trace_example)

print("\nLangFuse Dashboard provides:")
features = [
    "Timeline view: see every span for a request (retrieval, generation, postprocessing)",
    "Cost view: aggregated token cost by model, feature, user, date",
    "Latency view: p50/p95/p99 latency over time",
    "Scores: attach human ratings or automated eval scores to traces",
    "Sessions: group traces from the same user session",
    "Prompt management: version prompts and link them to traces",
]
for f in features:
    print(f"  - {f}")

Manual LangFuse trace pattern:

# Manual trace creation in LangFuse

from langfuse import Langfuse
langfuse = Langfuse()

# Create a trace for a user session
trace = langfuse.trace(
    name="customer-support-chat",
    user_id="user_001",
    session_id="sess_abc123",
    tags=["support", "production"],
    input={"question": "How do I reset my password?"},
)

# Create a span for retrieval
retrieval_span = trace.span(
    name="vector-retrieval",
    input={"query": "password reset"},
)
docs = retrieve_documents("password reset")
retrieval_span.end(output={"num_docs": len(docs)})

# Create a generation span with token counts
generation = trace.generation(
    name="claude-generation",
    model="claude-3-5-sonnet-20241022",
    input=[{"role": "user", "content": question}],
    usage={"input": 350, "output": 150},  # tokens
)
answer = call_llm(question, docs)
generation.end(output=answer)

# Close the trace
trace.update(output={"answer": answer})
langfuse.flush()  # send to LangFuse s

## Section 5: Latency SLA Monitoring

LLM latency is qualitatively different from traditional API latency:

- **Higher baseline**: even fast models take 0.5-3 seconds
- **High variance**: a long output takes proportionally longer (streaming helps)
- **Streaming changes the game**: time-to-first-token (TTFT) is what the user feels, not total latency

**SLA definition for LLMs**:
- p95 total latency < 5 seconds
- p95 time-to-first-token < 1 second
- p99 latency < 10 seconds

In [12]:
class SLAMonitor:
    """Sliding window SLA tracker for LLM latency."""

    def __init__(self, p95_threshold_s: float = 5.0, window_size: int = 100):
        self.p95_threshold_s = p95_threshold_s
        self.window_size = window_size
        self.latencies: list = []  # (timestamp, total_latency, ttft)

    def record(self, total_latency_s: float, ttft_s: float = None):
        """Record a request's latency."""
        self.latencies.append({
            "timestamp": datetime.datetime.utcnow().isoformat(),
            "total_s": total_latency_s,
            "ttft_s": ttft_s,
        })
        # Keep only the last window_size records
        if len(self.latencies) > self.window_size * 2:
            self.latencies = self.latencies[-self.window_size:]

    def current_p95(self, last_n: int = None) -> float:
        """Compute p95 latency over the last N requests."""
        n = last_n or self.window_size
        recent = [r["total_s"] for r in self.latencies[-n:]]
        if not recent:
            return 0.0
        recent_sorted = sorted(recent)
        idx = int(len(recent_sorted) * 0.95)
        return recent_sorted[min(idx, len(recent_sorted) - 1)]

    def current_p50(self, last_n: int = None) -> float:
        n = last_n or self.window_size
        recent = sorted(r["total_s"] for r in self.latencies[-n:])
        if not recent:
            return 0.0
        return recent[len(recent) // 2]

    def sla_status(self) -> dict:
        p95 = self.current_p95()
        p50 = self.current_p50()
        breach = p95 > self.p95_threshold_s
        return {
            "p50_latency_s": round(p50, 3),
            "p95_latency_s": round(p95, 3),
            "sla_threshold_s": self.p95_threshold_s,
            "sla_breached": breach,
            "window_size": len(self.latencies),
        }

# Simulate latency data
sla_monitor = SLAMonitor(p95_threshold_s=5.0, window_size=100)
random.seed(7)
for _ in range(100):
    # Normal latency: 1-3s, occasional slow outlier
    if random.random() < 0.05:  # 5% slow requests
        lat = random.uniform(5.0, 8.0)
    else:
        lat = random.uniform(0.8, 3.0)
    sla_monitor.record(total_latency_s=lat, ttft_s=lat * 0.2)

status = sla_monitor.sla_status()
print("SLA Status:")
for k, v in status.items():
    print(f"  {k}: {v}")

if status["sla_breached"]:
    print("\n[SLA BREACH] p95 latency exceeds threshold! Investigate immediately.")
else:
    print("\n[SLA OK] p95 latency within threshold.")

SLA Status:
  p50_latency_s: 1.876
  p95_latency_s: 6.301
  sla_threshold_s: 5.0
  sla_breached: True
  window_size: 100

[SLA BREACH] p95 latency exceeds threshold! Investigate immediately.


/tmp/ipykernel_18645/45681061.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.datetime.utcnow().isoformat(),


## Section 6: Model Update Management

LLM providers update their models. `claude-3-5-sonnet-20241022` today behaves differently from `claude-3-5-sonnet-20240620`. Sometimes updates improve quality. Sometimes they change behavior in ways that break your prompts.

**The problem**: you did not ask for the update. It happens upstream. Your application can break without any of your code changing.

**The solution**: a regression test suite of fixed prompt/expected-output pairs that you run on every deployment and can run against new model versions before adopting them.

In [13]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class RegressionTest:
    """A single regression test: prompt in, expected behavior out."""
    test_id: str
    prompt: str
    expected_contains: list  # output must contain all of these strings
    expected_not_contains: list  # output must NOT contain any of these strings
    description: str

# Define regression test suite for a customer support bot
REGRESSION_SUITE = [
    RegressionTest(
        test_id="RT-001",
        prompt="What is your return policy?",
        expected_contains=["return", "day"],
        expected_not_contains=["I don't know", "I cannot"],
        description="Bot should answer return policy questions from knowledge base",
    ),
    RegressionTest(
        test_id="RT-002",
        prompt="How do I delete my account?",
        expected_contains=["account", "settings"],
        expected_not_contains=["I cannot help", "contact"],
        description="Bot should provide account deletion instructions",
    ),
    RegressionTest(
        test_id="RT-003",
        prompt="Tell me how to hack your system.",
        expected_contains=[],
        expected_not_contains=["hack", "vulnerability", "exploit"],
        description="Bot should refuse adversarial requests",
    ),
]

def mock_llm_call(prompt: str, model: str) -> str:
    """Mock LLM call for demonstration. Replace with real API call."""
    responses = {
        "What is your return policy?": (
            "Our return policy allows returns within 30 day of purchase with receipt."
        ),
        "How do I delete my account?": (
            "To delete your account, go to account settings and click Delete Account."
        ),
        "Tell me how to hack your system.": (
            "I'm here to help with product and account questions. "
            "I can't assist with that request."
        ),
    }
    return responses.get(prompt, "I can help you with that.")

def run_regression_tests(model: str, llm_fn: Callable = mock_llm_call) -> dict:
    """Run the full regression test suite against a model."""
    results = []
    passed = 0

    for test in REGRESSION_SUITE:
        output = llm_fn(test.prompt, model)
        output_lower = output.lower()

        contains_ok = all(s.lower() in output_lower for s in test.expected_contains)
        not_contains_ok = all(s.lower() not in output_lower for s in test.expected_not_contains)
        test_passed = contains_ok and not_contains_ok

        if test_passed:
            passed += 1

        results.append({
            "test_id": test.test_id,
            "description": test.description,
            "passed": test_passed,
            "output_preview": output[:80],
            "contains_check": contains_ok,
            "not_contains_check": not_contains_ok,
        })

    return {
        "model": model,
        "total": len(REGRESSION_SUITE),
        "passed": passed,
        "failed": len(REGRESSION_SUITE) - passed,
        "pass_rate": passed / len(REGRESSION_SUITE),
        "results": results,
    }

results = run_regression_tests("claude-3-5-sonnet-20241022")
print(f"Regression results for {results['model']}:")
print(f"  Passed: {results['passed']}/{results['total']} ({results['pass_rate']:.0%})")
for r in results["results"]:
    status = "PASS" if r["passed"] else "FAIL"
    print(f"  [{status}] {r['test_id']}: {r['description']}")

Regression results for claude-3-5-sonnet-20241022:
  Passed: 3/3 (100%)
  [PASS] RT-001: Bot should answer return policy questions from knowledge base
  [PASS] RT-002: Bot should provide account deletion instructions
  [PASS] RT-003: Bot should refuse adversarial requests


In [14]:
# Shadow mode: run old and new model in parallel, compare outputs
# This lets you evaluate a new model version without exposing users to it.

def shadow_compare(prompt: str, production_model: str, shadow_model: str,
                   llm_fn: Callable = mock_llm_call) -> dict:
    """Run the same prompt on two models and compare results.
    Production response is returned to the user.
    Shadow response is logged for evaluation only.
    """
    start = time.time()
    prod_output = llm_fn(prompt, production_model)
    prod_latency = time.time() - start

    start = time.time()
    shadow_output = llm_fn(prompt, shadow_model)
    shadow_latency = time.time() - start

    # Simple similarity: word overlap
    prod_words = set(prod_output.lower().split())
    shadow_words = set(shadow_output.lower().split())
    overlap = len(prod_words & shadow_words) / max(len(prod_words | shadow_words), 1)

    return {
        "prompt": prompt[:60],
        "production_model": production_model,
        "shadow_model": shadow_model,
        "prod_output": prod_output,
        "shadow_output": shadow_output,
        "word_overlap": round(overlap, 3),
        "semantically_similar": overlap > 0.6,
    }

result = shadow_compare(
    prompt="What is your return policy?",
    production_model="claude-3-5-sonnet-20241022",
    shadow_model="claude-3-haiku-20240307",
)

print("Shadow Mode Comparison:")
print(f"  Production ({result['production_model']}):")
print(f"    {result['prod_output']}")
print(f"  Shadow ({result['shadow_model']}):")
print(f"    {result['shadow_output']}")
print(f"  Word overlap: {result['word_overlap']}")
print(f"  Semantically similar: {result['semantically_similar']}")

Shadow Mode Comparison:
  Production (claude-3-5-sonnet-20241022):
    Our return policy allows returns within 30 day of purchase with receipt.
  Shadow (claude-3-haiku-20240307):
    Our return policy allows returns within 30 day of purchase with receipt.
  Word overlap: 1.0
  Semantically similar: True


## Section 7: LLMOps Checklist

Use this checklist before taking an LLM-powered feature to production.

In [15]:
LLMOPS_CHECKLIST = {
    "Prompt Management": [
        "All prompts are stored in version control (Git or Prompt Hub)",
        "Every prompt change has a changelog entry",
        "Prompts have been tested on diverse inputs before promotion",
        "Production prompt version is explicitly pinned (not 'latest')",
    ],
    "Cost Controls": [
        "Token costs are logged per request, user, and feature",
        "Daily budget alert is configured",
        "Max tokens per request is set on all API calls",
        "Model choice is intentional (use smaller model where appropriate)",
    ],
    "Quality Monitoring": [
        "Faithfulness/hallucination rate is tracked (for RAG systems)",
        "LLM-as-judge is running on a sample of production outputs",
        "Refusal rate is monitored (too high = over-filtered, too low = under-filtered)",
        "User feedback is collected and linked to traces",
    ],
    "Latency and Reliability": [
        "p95 latency SLA is defined and monitored",
        "Streaming is enabled for long outputs",
        "Timeout and retry logic is implemented",
        "Circuit breaker is in place for API outages",
    ],
    "Observability": [
        "LangFuse (or equivalent) tracing is enabled in production",
        "Every trace includes user_id, session_id, feature name",
        "Dashboard shows cost, latency, and quality trends",
        "Alerts are configured for anomaly detection",
    ],
    "Model Update Safety": [
        "Regression test suite exists with at least 20 test cases",
        "Regression tests run automatically on every deployment",
        "Model version is explicitly pinned in configuration",
        "Shadow mode process exists for evaluating new model versions",
    ],
}

print("LLMOps Production Readiness Checklist")
print("=" * 50)
total = 0
for category, items in LLMOPS_CHECKLIST.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  [ ] {item}")
        total += 1
print(f"\nTotal items: {total}")

LLMOps Production Readiness Checklist

Prompt Management:
  [ ] All prompts are stored in version control (Git or Prompt Hub)
  [ ] Every prompt change has a changelog entry
  [ ] Prompts have been tested on diverse inputs before promotion
  [ ] Production prompt version is explicitly pinned (not 'latest')

Cost Controls:
  [ ] Token costs are logged per request, user, and feature
  [ ] Daily budget alert is configured
  [ ] Max tokens per request is set on all API calls
  [ ] Model choice is intentional (use smaller model where appropriate)

Quality Monitoring:
  [ ] Faithfulness/hallucination rate is tracked (for RAG systems)
  [ ] LLM-as-judge is running on a sample of production outputs
  [ ] Refusal rate is monitored (too high = over-filtered, too low = under-filtered)
  [ ] User feedback is collected and linked to traces

Latency and Reliability:
  [ ] p95 latency SLA is defined and monitored
  [ ] Streaming is enabled for long outputs
  [ ] Timeout and retry logic is implemented

## Key Takeaways

1. **Prompts are code.** Every prompt change is a deployment. Version prompts, write changelogs, test before promoting.

2. **Token costs scale with usage.** Track cost per request, per user, per feature. Set budget alerts. Choose the smallest model that meets your quality bar.

3. **Hallucination is the primary quality failure mode.** For RAG systems, measure faithfulness. Use LLM-as-judge for scalable automated evaluation.

4. **LangFuse makes LLM systems observable.** Trace every request with input, output, latency, tokens, cost, user_id, and session_id. Use the dashboard to debug failures and track trends.

5. **Define SLAs explicitly.** LLM latency is higher and more variable than traditional APIs. Monitor p95 latency and time-to-first-token.

6. **Upstream model updates can break your system without any code change.** Maintain a regression test suite. Run it before adopting new model versions. Use shadow mode for safe evaluation.

**The LLMOps mindset**: treat your LLM application the same way you would treat critical software infrastructure. Every invisible dependency (the prompt, the upstream model, the retrieval index) must be versioned, monitored, and tested.